[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C37_MLOps_Course/04_monitoring_drift/04_monitoring_drift.ipynb)

# 04 · 从零实现漂移检测器

本 notebook 从零造一个漂移检测器：**PSI 分箱 + KS/ECDF 两样本检验 + 滑动窗口 + 分级告警**，用 **numpy + pandas + 标准库** 实现，过 `assert`。核心是在**无标签**现实下用数据分布变化预警模型劣化。

**路线**：① 同分布 PSI≈0、漂移则↑ → ② PSI 的两个魔鬼细节（固定分箱+空箱平滑）→ ③ KS/ECDF → ④ 滑动窗口 → ⑤ 分级+持续性告警 → ✏️ 4 道练习 → 📖 答案 → 🧪 真实分布漂移下的端到端监控。

In [ ]:
import numpy as np
import pandas as pd
rng = np.random.default_rng(0)
print('环境就绪 ✅ | numpy', np.__version__, '| pandas', pd.__version__)

## 1 · PSI：同分布 ≈ 0，漂移则升高

PSI = Σ(p_i - q_i)·ln(p_i/q_i)，p/q 是当前/参考期各箱占比。先实现它，验证：**同分布 PSI≈0；分布越漂移 PSI 越大**。

In [ ]:
def psi(reference, current, bins=10, eps=1e-6):
    '''参考期定分箱边界，当前期套用同样边界，比较占比。'''
    # 在参考期上用分位数定箱边界（固定！）
    edges = np.quantile(reference, np.linspace(0, 1, bins + 1))
    edges[0], edges[-1] = -np.inf, np.inf            # 两端开放，兜住超出范围的新数据
    q = np.histogram(reference, edges)[0] / len(reference)   # 参考占比
    p = np.histogram(current, edges)[0] / len(current)      # 当前占比
    q = np.clip(q, eps, None); p = np.clip(p, eps, None)     # 空箱平滑，防 ln(0)/除0
    return float(np.sum((p - q) * np.log(p / q)))

ref = rng.normal(0, 1, 5000)
same = rng.normal(0, 1, 5000)                  # 同分布
shifted = rng.normal(0.5, 1, 5000)             # 均值漂移
big_shift = rng.normal(1.5, 1.5, 5000)         # 大漂移（均值+方差）
print(f'PSI(同分布)   = {psi(ref, same):.4f}   (期望 ≈0)')
print(f'PSI(中等漂移) = {psi(ref, shifted):.4f}   (期望 0.1~0.25)')
print(f'PSI(大漂移)   = {psi(ref, big_shift):.4f}   (期望 >0.25)')
assert psi(ref, same) < 0.1, '同分布 PSI 应 < 0.1'
assert psi(ref, big_shift) > psi(ref, shifted) > psi(ref, same), 'PSI 应随漂移幅度单调增'
assert psi(ref, big_shift) > 0.25, '大漂移 PSI 应 > 0.25（显著）'
print('✅ PSI 正确：同分布≈0，漂移越大 PSI 越大')

## 2 · PSI 的两个魔鬼细节

**细节 A：分箱必须在参考期定一次、之后固定**。固定的尺子才能跨期比较——固定分箱下 PSI 随漂移幅度**单调上升**、各期可比；若每期用当期数据重新分箱（移动的尺子），PSI 失去单调性与可比性，无法说「这月比上月漂得更多」。
**细节 B：空箱平滑**。某箱占比为 0 时 ln(p/q) 会炸成 inf；加 eps 兜底。

In [ ]:
# 细节 A：固定分箱 -> PSI 随漂移幅度单调上升、可跨期比较
def psi_rebins_each_period(reference, current, bins=10, eps=1e-6):
    # 错误做法：用 current 自己的分位数分箱 -> 尺子每期都在动
    edges = np.quantile(current, np.linspace(0, 1, bins + 1))
    edges[0], edges[-1] = -np.inf, np.inf
    q = np.clip(np.histogram(reference, edges)[0] / len(reference), eps, None)
    p = np.clip(np.histogram(current, edges)[0] / len(current), eps, None)
    return float(np.sum((p - q) * np.log(p / q)))

shifts = [0.0, 0.3, 0.6, 1.0, 1.5]
fixed = [psi(ref, rng.normal(s, 1, 5000)) for s in shifts]               # 固定分箱
moving = [psi_rebins_each_period(ref, rng.normal(s, 1, 5000)) for s in shifts]  # 移动分箱
print('漂移幅度 :', shifts)
print('固定分箱 :', [round(v, 3) for v in fixed], ' <- 单调递增，可比')
print('移动分箱 :', [round(v, 3) for v in moving], ' <- 大小取决于动的尺子，不可比')
# 固定分箱：PSI 随漂移幅度单调上升（这正是它能当「漂移温度计」的前提）
assert all(fixed[i] < fixed[i+1] for i in range(len(fixed)-1)), '固定分箱下 PSI 应随漂移单调上升'
assert fixed[0] < 0.1 and fixed[-1] > 0.25, '从稳定到显著漂移，固定分箱给出可比的刻度'
print('✅ 细节 A：分箱必须在参考期固定 —— 固定的尺子才能跨期比较漂移')

In [ ]:
# 细节 B：演示不平滑会炸成 inf
def psi_no_smoothing(reference, current, bins=10):
    edges = np.quantile(reference, np.linspace(0, 1, bins + 1))
    edges[0], edges[-1] = -np.inf, np.inf
    q = np.histogram(reference, edges)[0] / len(reference)
    p = np.histogram(current, edges)[0] / len(current)
    return float(np.sum((p - q) * np.log(p / q)))      # 无 eps -> 可能 ln(0)

# 当前期完全不覆盖参考期的某些箱 -> 出现 0 占比
narrow = rng.normal(5, 0.1, 2000)               # 远离参考分布，很多箱占比为0
import warnings; warnings.filterwarnings('ignore')
val_no_smooth = psi_no_smoothing(ref, narrow)
val_smooth = psi(ref, narrow)
print(f'不平滑 PSI = {val_no_smooth}  (inf 或 nan)')
print(f'平滑后 PSI = {val_smooth:.4f}  (有限大数，正确反映剧烈漂移)')
assert not np.isfinite(val_no_smooth), '不平滑应炸成 inf/nan'
assert np.isfinite(val_smooth) and val_smooth > 0.25, '平滑后应是有限的大 PSI'
print('✅ 细节 B：空箱平滑必不可少，否则 PSI 炸成 inf')

## 3 · KS 检验：免分箱的 ECDF 最大距离

ECDF: F̂(x)=(≤x 的样本比例)。KS 统计量 D = 两条 ECDF 的最大纵向距离。无需分箱参数。

捷径：合并排序两组样本，在每个点算两 ECDF 之差，取 |最大|。再用近似公式给 p 值。

In [ ]:
def ks_statistic(a, b):
    '''两样本 KS 统计量 D = sup|F_a - F_b|，O((n+m)log(n+m)).'''
    a = np.sort(a); b = np.sort(b)
    allv = np.concatenate([a, b])
    # 在每个数据点处算两条 ECDF
    cdf_a = np.searchsorted(a, allv, side='right') / len(a)
    cdf_b = np.searchsorted(b, allv, side='right') / len(b)
    return float(np.max(np.abs(cdf_a - cdf_b)))

def ks_pvalue_approx(D, n, m):
    '''Kolmogorov 渐近 p 值。'''
    ne = n * m / (n + m)
    lam = (np.sqrt(ne) + 0.12 + 0.11 / np.sqrt(ne)) * D
    # 渐近分布 Q(lam) = 2 Σ (-1)^{k-1} exp(-2 k^2 lam^2)
    s = sum((-1)**(k-1) * np.exp(-2 * k**2 * lam**2) for k in range(1, 101))
    return float(max(0.0, min(1.0, 2 * s)))

D_same = ks_statistic(ref, same)
D_shift = ks_statistic(ref, big_shift)
print(f'KS D(同分布) = {D_same:.4f}, p = {ks_pvalue_approx(D_same, len(ref), len(same)):.3f}')
print(f'KS D(大漂移) = {D_shift:.4f}, p = {ks_pvalue_approx(D_shift, len(ref), len(big_shift)):.3g}')
assert D_same < D_shift, 'KS 距离应随漂移增大'
assert ks_pvalue_approx(D_shift, len(ref), len(big_shift)) < 0.01, '大漂移应显著 (p<0.01)'
assert ks_pvalue_approx(D_same, len(ref), len(same)) > 0.01, '同分布不应显著'
print('✅ KS 检验正确：D 随漂移增大，p 值能区分同分布与漂移')

In [ ]:
# 与 scipy 对拍（若可用）——证明我们从零实现的 KS 是对的
try:
    from scipy.stats import ks_2samp
    D_ours = ks_statistic(ref, big_shift)
    D_scipy = ks_2samp(ref, big_shift).statistic
    print(f'我们的 D = {D_ours:.5f} | scipy D = {D_scipy:.5f}')
    assert abs(D_ours - D_scipy) < 1e-6, '应与 scipy 逐位一致'
    print('✅ 与 scipy.stats.ks_2samp 对拍一致')
except ImportError:
    print('（无 scipy，跳过对拍；我们的实现已通过上面的 assert）')

## 4 · 滑动窗口：用最近数据估当前分布

漂移检测要「参考期 vs 当前期」。当前期 = 最近一个窗口的数据。窗口大小是「灵敏 vs 稳定」的权衡。

模拟一条**逐渐漂移**的数据流，用滑窗逐步算 PSI，看它何时越过告警线。

In [ ]:
def streaming_psi(stream, reference, window=500, step=250, bins=10):
    '''在数据流上滑窗算 PSI 序列，返回 [(位置, psi), ...].'''
    out = []
    for start in range(0, len(stream) - window + 1, step):
        win = stream[start:start + window]
        out.append((start + window, psi(reference, win, bins=bins)))
    return out

# 数据流：前半稳定(均值0)，后半逐渐漂移到均值2
ref_w = rng.normal(0, 1, 5000)
n_stream = 6000
drift_amount = np.concatenate([np.zeros(n_stream//2),
                               np.linspace(0, 2, n_stream - n_stream//2)])
stream = rng.normal(0, 1, n_stream) + drift_amount

series = streaming_psi(stream, ref_w, window=500, step=250)
positions = [pos for pos, _ in series]
psis = [val for _, val in series]
print('滑窗 PSI 轨迹（位置: PSI）：')
for pos, val in series[::3]:
    bar = '#' * int(val * 40)
    print(f'  pos={pos:5d}: {val:.3f} {bar}')
# 前期 PSI 低，后期随漂移升高并越过 0.25
assert psis[0] < 0.1, '稳定期 PSI 应低'
assert max(psis) > 0.25, '漂移期 PSI 应越过显著线'
assert psis[-1] > psis[0], 'PSI 整体随漂移上升'
print('✅ 滑窗 PSI 捕捉到渐变漂移：稳定期低、漂移期升高越线')

## 5 · 分级 + 持续性告警：别叫狼来了

直接「PSI 超 0.25 就告警」会被噪声反复触发。实战告警要：**分级**（警惕/告警）+ **持续性**（连续 k 个窗口超阈才真告警）。

In [ ]:
def alerting(psi_series, warn=0.1, alert=0.25, persistence=2):
    '''分级 + 持续性告警。返回每个窗口的状态与是否触发正式告警。'''
    states = []
    consec = 0                    # 连续超 alert 阈的窗口数
    for pos, val in psi_series:
        if val >= alert:
            consec += 1
            level = 'ALERT' if consec >= persistence else 'warn(超阈但未持续)'
        elif val >= warn:
            consec = 0; level = 'warn'
        else:
            consec = 0; level = 'ok'
        fired = (level == 'ALERT')
        states.append((pos, round(val, 3), level, fired))
    return states

states = alerting(series, warn=0.1, alert=0.25, persistence=2)
print('告警轨迹（位置, PSI, 级别, 是否正式告警）：')
for s in states[::2]:
    print('  ', s)
fired_positions = [pos for (pos, v, lvl, fired) in states if fired]
assert fired_positions, '漂移持续后应触发正式告警'
# 第一次正式告警必然在「连续 persistence 个窗口超阈」之后，而非一超就报
first_over = next(pos for (pos, v, lvl, fired) in states if v >= 0.25)
first_fire = fired_positions[0]
assert first_fire > first_over, '正式告警应晚于首次超阈（持续性要求过滤了瞬时尖峰）'
print(f'\n首次超阈 @pos={first_over}，但正式告警 @pos={first_fire}（持续性生效）')
print('✅ 分级+持续性告警：过滤瞬时噪声，只在持续漂移时叫人')

---
## ✏️ 练习 1：类别变量的 PSI（χ²-style）

PSI 也能用于**类别变量**（不分箱，每个类别就是一个箱）。实现 `categorical_psi(ref_labels, cur_labels, eps=1e-6)`：
对两组类别标签，按每个类别的占比算 PSI（公式同连续版，箱=类别）。

In [ ]:
def categorical_psi(ref_labels, cur_labels, eps=1e-6):
    # TODO: 取所有出现过的类别；分别算 ref/cur 中每个类别的占比 q_i/p_i；
    #       clip 到 eps 后算 sum((p-q)*ln(p/q))。返回 float。
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
ref_lab = np.array(['a']*600 + ['b']*300 + ['c']*100)
same_lab = np.array(['a']*590 + ['b']*310 + ['c']*100)        # 几乎同分布
drift_lab = np.array(['a']*200 + ['b']*300 + ['c']*500)       # c 类大增
assert categorical_psi(ref_lab, same_lab) < 0.1, '几乎同分布 PSI 应低'
assert categorical_psi(ref_lab, drift_lab) > 0.25, 'c 类大增应是显著漂移'
assert categorical_psi(ref_lab, ref_lab) < 1e-9, '完全相同 PSI≈0'
print('cat PSI(同分布)=%.4f, (漂移)=%.4f' % (categorical_psi(ref_lab,same_lab), categorical_psi(ref_lab,drift_lab)))
print('✅ 练习 1 通过：类别变量 PSI')

## ✏️ 练习 2：从零实现 ECDF 并验证 KS 的几何意义

实现 `ecdf(data, x)`：返回经验累积分布在点 `x`（标量或数组）处的值 =（≤x 的样本比例）。
然后用它实现 `ks_via_ecdf(a, b)`：在 a∪b 的所有点上算 |ecdf_a - ecdf_b| 的最大值（应等于 worked 里的 `ks_statistic`）。

In [ ]:
def ecdf(data, x):
    # TODO: 返回 (data <= x 的比例)。x 可为标量或 np 数组；用 np.searchsorted 高效实现
    raise NotImplementedError

def ks_via_ecdf(a, b):
    # TODO: 在 concatenate([a,b]) 的每个点上算 |ecdf(a,·)-ecdf(b,·)|，返回最大值
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
d = np.array([1.0, 2.0, 3.0, 4.0])
assert abs(ecdf(d, 2.5) - 0.5) < 1e-9, 'ecdf: <=2.5 的有 2/4=0.5'
assert abs(ecdf(d, 0.0) - 0.0) < 1e-9 and abs(ecdf(d, 5.0) - 1.0) < 1e-9
assert np.allclose(ecdf(d, np.array([1.0, 4.0])), [0.25, 1.0]), 'ecdf 支持数组'
# ks_via_ecdf 应与 worked 的 ks_statistic 完全一致
a, b = rng.normal(0,1,500), rng.normal(0.7,1,500)
assert abs(ks_via_ecdf(a, b) - ks_statistic(a, b)) < 1e-9, '两种实现应一致'
print('✅ 练习 2 通过：ECDF 正确，KS = 两 ECDF 最大间距')

## ✏️ 练习 3：自适应窗口的雏形——按 PSI 选窗口

固定窗口要权衡。一个简单的自适应思路：**漂移大就缩短窗口（快反应），漂移小就放长（求稳）**。
实现 `adaptive_window(psi_value, w_min=200, w_max=1000)`：PSI ≥ 0.25 返回 `w_min`，PSI ≤ 0.1 返回 `w_max`，中间线性插值（PSI 越大窗口越小）。返回 int。

In [ ]:
def adaptive_window(psi_value, w_min=200, w_max=1000):
    # TODO: psi>=0.25 -> w_min; psi<=0.1 -> w_max; 中间按 psi 线性插值（单调递减）
    #       提示：t=(psi-0.1)/(0.25-0.1) 截断到[0,1]，w = w_max + t*(w_min-w_max)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
assert adaptive_window(0.30) == 200, '大漂移 -> 最小窗口（快反应）'
assert adaptive_window(0.05) == 1000, '稳定 -> 最大窗口（求稳）'
mid = adaptive_window(0.175)            # 正中间
assert 550 <= mid <= 650, f'中间漂移应得中等窗口, 得到 {mid}'
# 单调性：PSI 越大，窗口越小
assert adaptive_window(0.2) < adaptive_window(0.15), 'PSI 越大窗口越小'
print(f'PSI=0.30->{adaptive_window(0.30)}, 0.175->{mid}, 0.05->{adaptive_window(0.05)}')
print('✅ 练习 3 通过：自适应窗口雏形（漂移大则缩窗快反应）')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def categorical_psi(ref_labels, cur_labels, eps=1e-6):
    cats = set(ref_labels) | set(cur_labels)
    nref, ncur = len(ref_labels), len(cur_labels)
    total = 0.0
    for cat in cats:
        q = max((ref_labels == cat).sum() / nref, eps)
        p = max((cur_labels == cat).sum() / ncur, eps)
        total += (p - q) * np.log(p / q)
    return float(total)

In [ ]:
# 练习 2 参考答案
def ecdf(data, x):
    s = np.sort(data)
    return np.searchsorted(s, x, side='right') / len(s)

def ks_via_ecdf(a, b):
    pts = np.concatenate([a, b])
    return float(np.max(np.abs(ecdf(a, pts) - ecdf(b, pts))))

In [ ]:
# 练习 3 参考答案
def adaptive_window(psi_value, w_min=200, w_max=1000):
    t = (psi_value - 0.1) / (0.25 - 0.1)
    t = min(1.0, max(0.0, t))
    return int(round(w_max + t * (w_min - w_max)))

---
## 🧪 真实数据胶囊：真实分布漂移下的端到端监控

构造一个**贴近真实**的场景：一个信贷风控模型，参考期是训练时的申请人分布；上线后，因经济环境变化，申请人的「收入」分布逐月漂移、「负债比」分布也变。我们用 PSI + KS 同时监控这两个特征，并验证「漂移导致模型 AUC 下滑」。

In [ ]:
# 申请人特征：收入、负债比。w_inc 控制「收入->违约」的真实关系强度（概念漂移用）
def make_applicants(n, income_mean, dti_mean, seed, w_inc=1.0):
    r = np.random.default_rng(seed)
    income = r.lognormal(income_mean, 0.4, n)
    dti = np.clip(r.normal(dti_mean, 0.1, n), 0, 1)        # debt-to-income
    zi = (income - np.exp(10.5)) / np.exp(10.5)            # 用固定参考尺度，避免每期自归一
    # 真实违约：收入低、负债高 -> 更易违约；w_inc 越小，收入这个信号越弱（概念漂移）
    score = -w_inc * zi + 4.0 * (dti - 0.35)
    default = (r.random(n) < 1/(1+np.exp(-score))).astype(int)
    return pd.DataFrame({'income': income, 'dti': dti, 'default': default})

ref_df = make_applicants(5000, income_mean=10.5, dti_mean=0.35, seed=100, w_inc=1.0)
# 风控模型：用参考期固定的归一化尺度打分（上线后尺度不再更新——这正是真实情况）
REF_INC_MEAN, REF_INC_STD = ref_df['income'].mean(), ref_df['income'].std()
REF_DTI_MEAN, REF_DTI_STD = ref_df['dti'].mean(), ref_df['dti'].std()
def scorer(df):
    zi = (df['income'] - REF_INC_MEAN) / REF_INC_STD
    zd = (df['dti'] - REF_DTI_MEAN) / REF_DTI_STD
    return (-1.0 * zi + 1.0 * zd).values            # 固定权重：低收入/高负债 -> 高风险分

def auc(scores, labels):
    # Mann-Whitney AUC：随机正负对中正样本得分更高的概率
    order = np.argsort(scores); ranks = np.empty(len(scores), dtype=float)
    ranks[order] = np.arange(1, len(scores)+1)
    n_pos = int(labels.sum()); n_neg = len(labels) - n_pos
    if n_pos == 0 or n_neg == 0: return 0.5
    return float((ranks[labels==1].sum() - n_pos*(n_pos+1)/2) / (n_pos*n_neg))

auc_ref = auc(scorer(ref_df), ref_df['default'].values)
print(f'参考期 AUC = {auc_ref:.3f}')
assert auc_ref > 0.6, '模型在参考期应有效'

In [ ]:
# 上线后逐月：收入下降+负债上升(数据漂移) 且 收入信号减弱(概念漂移 w_inc 下降)
print(f"{'月份':<6}{'PSI(收入)':>12}{'PSI(负债比)':>14}{'KS(收入)':>12}{'AUC':>10}{'告警':>8}")
schedule = [(10.5,0.35,1.0), (10.4,0.38,0.7), (10.2,0.42,0.4), (10.0,0.48,0.15)]
auc_list = []
for month, (inc_m, dti_m, w) in enumerate(schedule):
    cur = make_applicants(4000, income_mean=inc_m, dti_mean=dti_m, seed=200+month, w_inc=w)
    psi_inc = psi(ref_df['income'].values, cur['income'].values)
    psi_dti = psi(ref_df['dti'].values, cur['dti'].values)
    ks_inc = ks_statistic(ref_df['income'].values, cur['income'].values)
    auc_cur = auc(scorer(cur), cur['default'].values)
    auc_list.append(auc_cur)
    alert = 'ALERT' if max(psi_inc, psi_dti) > 0.25 else ('warn' if max(psi_inc,psi_dti) > 0.1 else 'ok')
    print(f'{month:<6}{psi_inc:>12.4f}{psi_dti:>14.4f}{ks_inc:>12.4f}{auc_cur:>10.3f}{alert:>8}')

# 监控的价值：数据漂移可观测(无标签)，且最终 AUC 确实下滑(有标签才知道)
assert psi(ref_df['dti'].values, make_applicants(4000,10.0,0.48,999,0.15)['dti'].values) > 0.1, '末期负债比应明显漂移'
assert auc_list[-1] < auc_list[0] - 0.03, '漂移最终导致 AUC 下滑（监控预警是对的）'
print('\n✅ 端到端监控跑通：PSI/KS 升高(无标签可见) + AUC 最终下滑(需标签确认) —— 预警生效')

**🧪 胶囊练习**：实现 `monitor_features(ref_df, cur_df, features, psi_thresh=0.25)`：对给定特征列表，返回一个 dict `{特征名: (psi值, 是否告警)}`，告警 = PSI ≥ 阈值。这就是一个多特征监控面板的内核。

In [ ]:
def monitor_features(ref_df, cur_df, features, psi_thresh=0.25):
    # TODO: 对每个特征算 psi(ref_df[f], cur_df[f])，返回 {f: (psi值, psi>=阈值)}
    raise NotImplementedError

In [ ]:
# 自测
drifted = make_applicants(3000, 10.0, 0.48, seed=777)
panel = monitor_features(ref_df, drifted, ['income', 'dti'])
assert set(panel.keys()) == {'income', 'dti'}
assert all(isinstance(v[0], float) and isinstance(v[1], (bool, np.bool_)) for v in panel.values())
# 大漂移下至少一个特征告警
assert any(fired for _, fired in panel.values()), '大漂移下应至少一个特征告警'
print('监控面板:', {k: (round(v[0],3), bool(v[1])) for k,v in panel.items()})
print('✅ 胶囊练习通过：多特征监控面板')

In [ ]:
# 📖 胶囊参考答案
def monitor_features(ref_df, cur_df, features, psi_thresh=0.25):
    out = {}
    for f in features:
        v = psi(ref_df[f].values, cur_df[f].values)
        out[f] = (v, v >= psi_thresh)
    return out

---
## 🔧 旁注：这套东西对应 Evidently / NannyML 的什么

你写的漂移检测器对应生产监控库的核心（伪代码，**本环境不跑**）：

```python
from evidently.report import Report
from evidently.metrics import DataDriftTable
report = Report(metrics=[DataDriftTable()])     # 内部就是逐特征 PSI/KS/χ²
report.run(reference_data=ref_df, current_data=cur_df)
# == 我们的 monitor_features：逐特征 psi()/ks_statistic() + 阈值告警
# NannyML 更进一步：用置信度结构估计无标签下的性能（我们 frontier 提到的 CBPE）
```

对应关系：`DataDriftTable`↔`monitor_features`、PSI/KS↔我们从零实现的两个度量、告警阈值↔`alerting`。生产库多出来的是：内置多种检验、多重检验校正、可视化报告、与数据管道集成——但**统计内核与你写的一模一样**。

### 小结
- 上线后**没有标签**：用**数据漂移**当性能下滑的早期、无标签预警（必要预警，非充分证据）。
- **PSI**=分箱占比差的加权和；两个魔鬼细节：**分箱在参考期固定** + **空箱平滑**(防 inf)。
- **KS**=两条 ECDF 的最大距离，免分箱、带 p 值，但大样本过敏感；与 PSI 互补。
- **滑窗**权衡灵敏 vs 稳定；**告警**要分级 + 持续性，第一原则是别叫狼来了（防 alert fatigue）。
- 三层监控栈：数据漂移→预测漂移→性能监控（时效递减、可信递增）。

下一站：**模块 05 · 反馈闭环与重训练** —— 世界变了，模型该什么时候、怎么重训？会不会越学越忘？